# 4. Figures and tables for the main article

Collects every figure and table of the main article (primary analyses and registered secondary aim on streams and triage categories) and its supplement into `outputs/article/`, from the saved outputs of notebooks 01–03. Nothing is refitted here, except descriptive statistics of the panel (Table 1) and figures redrawn from saved estimates (Figures 1 and 4, Figure S7).

Content follows the manuscript outline (main paper: temperature, ILI, PM10 and PM2.5 as the four primary analyses). Exploratory analyses (holidays, occupancy, pollen, moon) belong to other papers and are **not** collected.

```
outputs/article/
├── README.md              index with draft captions and the source of each item
├── article_tables.xlsx    every table, one sheet each
├── main/                  Figures 1–4, Tables 1–2 (PNG at 300 dpi, CSV)
└── supplement/            Figures S1–S11, Tables S1–S11, deviations log
```

The folder is emptied and rebuilt on each run. The run stops if notebooks 03_1, 03_2 and 03_3 were not run on the current panel.

## Imports and configuration

In [ ]:
import hashlib
import textwrap
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
PANEL = ROOT / "data" / "processed" / "daily_panel_lagged.parquet"
CODEBOOK = ROOT / "data" / "processed" / "daily_panel_codebook.csv"
CCF = ROOT / "outputs" / "03_1_cross_correlation"
DLNM = ROOT / "outputs" / "03_2_dlnm"
CMP = ROOT / "outputs" / "03_3_comparison"
DEVIATIONS = ROOT / "docs" / "deviations_log.md"
OUT = ROOT / "outputs" / "article"
MAIN, SUPP = OUT / "main", OUT / "supplement"

COLORS = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "grey": "#8a8984", "red": "#e34948", "neutral": "#f0efec",
          "ink": "#1a1a1a"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False, "font.size": 9,
})
ANALYSES = {"TEMP": "Temperature → all arrivals", "ILI": "ILI → medical arrivals",
            "PM10": "PM10 → medical arrivals", "PM25": "PM2.5 → medical arrivals"}
INDEX = []   # (part, item, file, caption, source)


def fmt_p(p) -> str:
    if pd.isna(p):
        return ""
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def fmt_pcal(p) -> str:
    """Calibrated p from 1,000 dummies: smallest possible value 1/1001, so never '< 0.001'."""
    return "" if pd.isna(p) else f"{p:.3f}"


def rr_ci(rr, lo, hi, d=3) -> str:
    return "" if pd.isna(rr) else f"{rr:.{d}f} ({lo:.{d}f}–{hi:.{d}f})"


def register(part, item, path, caption, source):
    INDEX.append({"part": part, "item": item, "file": str(path.relative_to(OUT)), "caption": caption, "source": source})


def copy_fig(part, item, src, name, caption):
    dst = (MAIN if part == "main" else SUPP) / name
    shutil.copy2(src, dst)
    register(part, item, dst, caption, str(src.relative_to(ROOT)))


def save_table(part, item, df, name, caption, source, index=False):
    dst = (MAIN if part == "main" else SUPP) / name
    df.to_csv(dst, index=index)
    register(part, item, dst, caption, source)
    TABLES[item] = df.reset_index() if index else df


TABLES = {}
if OUT.exists():
    shutil.rmtree(OUT)
MAIN.mkdir(parents=True); SUPP.mkdir(parents=True)

## 1. Check that all inputs come from the same panel

In [ ]:
sha = hashlib.sha256(PANEL.read_bytes()).hexdigest()
runs = {name: json.loads((d / "run_info.json").read_text()) for name, d in (("03_1", CCF), ("03_2", DLNM), ("03_3", CMP))}
for name in ("03_1", "03_2"):
    if runs[name]["input_panel_sha256"] != sha:
        raise RuntimeError(f"Notebook {name} was run on a different panel: run notebooks 03_1-03_3 again.")
print(f"Panel {sha[:12]}…; 03_1 run {runs['03_1']['run_at']}, 03_2 run {runs['03_2']['run_at']}, 03_3 run {runs['03_3']['run_at']}")
panel = pd.read_parquet(PANEL)
codebook = pd.read_csv(CODEBOOK, index_col="column")
prim = pd.read_csv(DLNM / "primary_results.csv", index_col=0)
neg = pd.read_csv(DLNM / "negative_controls.csv", index_col=0)
sens = pd.read_csv(DLNM / "sensitivity.csv", index_col=0)
settings = runs["03_2"]["settings"]

## 2. Main article

### Table 1. Daily arrivals and exposures

Study days with arrival data. Arrivals are episodes (a referred patient counts once). Triage shares are of episodes with a category.

In [ ]:
days = panel["arr_all_mts_all"].notna()
p = panel.loc[days]
STREAMS = {"All arrivals": "all", "Injuries (KPP)": "injury", "Medical (Enota za bolezni + INP)": "medical",
           "  Enota za bolezni": "disease", "  INP": "inp", "Fast track (Hitri pregledi)": "fasttrack"}
rows = []
for lab, s in STREAMS.items():
    y = p[f"arr_{s}_mts_all"]
    tri = p[[f"arr_{s}_mts{k}" for k in range(1, 6)]].sum()
    share = 100 * tri / tri.sum()
    rows.append({"Variable": lab, "Unit": "episodes/day", "Days": int(y.notna().sum()), "Total": f"{int(y.sum()):,}",
                 "Mean (SD)": f"{y.mean():.1f} ({y.std():.1f})", "Median (range)": f"{y.median():.0f} ({y.min():.0f}–{y.max():.0f})",
                 "Triage 1/2/3/4/5, %": " / ".join(f"{v:.1f}" for v in share), "1st–99th percentile": ""})
EXPO = {"Daily mean temperature": ("temp_mean", "°C"), "ILI consultation rate (weekly)": ("ili_rate", "per 100,000"),
        "PM10": ("pm10", "µg/m³"), "PM2.5": ("pm25", "µg/m³")}
for lab, (c, unit) in EXPO.items():
    x = p[c]
    rows.append({"Variable": lab, "Unit": unit, "Days": int(x.notna().sum()), "Total": "",
                 "Mean (SD)": f"{x.mean():.1f} ({x.std():.1f})",
                 "Median (range)": f"{x.median():.1f} ({x.min():.1f}–{x.max():.1f})",
                 "Triage 1/2/3/4/5, %": "", "1st–99th percentile": f"{x.quantile(.01):.1f}–{x.quantile(.99):.1f}"})
table1 = pd.DataFrame(rows)
period = f"{p.index.min():%d %b %Y} – {p.index.max():%d %b %Y}"
save_table("main", "Table 1", table1, "Table_1_descriptives.csv",
           f"Daily ED arrivals by stream and the primary exposures, UKC Maribor, {period} ({int(days.sum()):,} days). "
           "Triage shares are of episodes with a category. ILI starts in June 2022 (ERVISS).",
           "data/processed/daily_panel_lagged.parquet")
display(table1)

### Figure 1. Time series of arrivals and primary exposures

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True, gridspec_kw={"hspace": 0.25})
covid_end = pd.Timestamp("2023-03-31")
y = p["arr_all_mts_all"]
ax = axes[0]
ax.plot(y.index, y, color=COLORS["grey"], lw=0.4, alpha=0.6, label="daily")
ax.plot(y.index, y.rolling(28, center=True, min_periods=14).mean(), color=COLORS["blue"], lw=1.6, label="28-day mean")
for d, lab in ((pd.Timestamp("2022-11-19"), "referral change"), (pd.Timestamp("2024-06-07"), "triage change")):
    ax.axvline(d, color=COLORS["ink"], ls=":", lw=0.8)
    ax.text(d, ax.get_ylim()[1], f" {lab}", fontsize=7, va="top")
ax.set_ylabel("ED arrivals\n(episodes/day)"); ax.legend(fontsize=7, loc="lower right", ncol=2)
ax = axes[1]
ax.plot(p.index, p["temp_mean"], color=COLORS["red"], lw=0.6)
ax.set_ylabel("Mean temperature\n(°C)")
ax = axes[2]
ili = p["ili_rate"]
ax.step(ili.index, ili, color=COLORS["aqua"], lw=0.9, where="post")
ax.set_ylabel("ILI rate\n(per 100,000)")
ax = axes[3]
ax.plot(p.index, p["pm10"], color=COLORS["ink"], lw=0.5, label="PM10")
ax.plot(p.index, p["pm25"], color=COLORS["orange"], lw=0.5, label="PM2.5")
ax.set_ylabel("Particulate matter\n(µg/m³)"); ax.legend(fontsize=7, loc="upper right", ncol=2)
for ax in axes:
    ax.axvspan(p.index.min(), covid_end, color=COLORS["neutral"], zorder=0, lw=0)
axes[0].text(p.index.min(), axes[0].get_ylim()[0], " COVID-19 period (to 31 Mar 2023)", fontsize=7, va="bottom", color=COLORS["grey"])
axes[-1].xaxis.set_major_locator(mdates.YearLocator()); axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
f1 = MAIN / "Figure_1_time_series.png"
fig.savefig(f1, bbox_inches="tight"); plt.show()
register("main", "Figure 1", f1, "Daily ED arrivals (grey: daily, blue: 28-day mean) and the primary exposures, UKC Maribor. "
         "Shading: COVID-19 period; dotted lines: structural breaks in the ED data (adjusted for in the models).",
         "data/processed/daily_panel_lagged.parquet")

### Figure 2. Temperature and all arrivals

In [ ]:
copy_fig("main", "Figure 2", DLNM / "TEMP_temperature.png", "Figure_2_temperature.png",
         "Temperature and all ED arrivals (DLNM). Top left: cumulative relative risk over lags 0–21 against the median "
         "temperature, with the temperature distribution; top right: lag–response at the 99th and 1st percentile; "
         "bottom: relative risk by temperature and lag. Bands: 95% CI widened by √κ (dummy-exposure calibration).")

### Table 2. Primary analyses, negative controls and future exposure

In [ ]:
rows = []
t = prim.loc["TEMP"]
heat, cold, med = t["heat_value"], t["cold_value"], t["median_temp"]
for lab, k in ((f"Heat, same day ({heat:.1f} vs {med:.1f} °C)", "heat_lag0"), (f"Cold, same day ({cold:.1f} vs {med:.1f} °C)", "cold_lag0"),
               ("Heat, cumulative lags 0–21", "heat_vs_median"), ("Cold, cumulative lags 0–21", "cold_vs_median")):
    rows.append({"Analysis": ANALYSES["TEMP"], "Contrast": lab, "RR (95% CI)": rr_ci(t[f"RR_{k}"], t[f"lo_{k}"], t[f"hi_{k}"])})
for h in ("ILI", "PM10", "PM25"):
    r = prim.loc[h]
    contrast = "10th → 90th percentile, lags 0–14" if h == "ILI" else "per 10 µg/m³, lags 0–3"
    rows.append({"Analysis": ANALYSES[h], "Contrast": contrast, "RR (95% CI)": rr_ci(r["RR"], r["lo"], r["hi"])})
table2 = pd.DataFrame(rows)
for h in ANALYSES:
    first = table2.index[table2["Analysis"] == ANALYSES[h]][0]
    r = prim.loc[h]
    table2.loc[first, "Days"] = f"{int(r['n_days']):,}"
    table2.loc[first, "κ"] = f"{r['kappa']:.2f}"
    table2.loc[first, "p nominal"] = fmt_p(r["p"])
    table2.loc[first, "p calibrated"] = fmt_pcal(r["p calibrated"])
    table2.loc[first, "p calibrated, Holm"] = fmt_pcal(r["p calibrated, Holm"])
    table2.loc[first, "Future exposure, p calibrated"] = fmt_pcal(neg.loc[f"{h} + future exposure", "p_lead calibrated"])
    if h != "TEMP":
        for nc, lab in (("KPP", "Injuries (unit), RR (95% CI)"), ("ICD S/T", "Injuries (diagnosis), RR (95% CI)")):
            n = neg.loc[f"{h} → injuries ({nc})"]
            table2.loc[first, lab] = rr_ci(n["RR"], n["lo"], n["hi"])
table2 = table2.astype(object).fillna("")
save_table("main", "Table 2", table2, "Table_2_primary_results.csv",
           "Primary analyses (quasi-Poisson DLNM). RR with 95% CI widened by √κ; p-values from the F-test of the cross-basis: nominal, "
           "calibrated against 1,000 dummy exposures, and calibrated with Holm correction across the four primary analyses. "
           "Negative-control outcomes (injuries by unit and by diagnosis; nominal 95% CI) and future exposure (calibrated p) test for "
           "residual confounding; they do not apply to temperature, which affects injuries directly.",
           "outputs/03_2_dlnm/primary_results.csv, negative_controls.csv")
display(table2)

### Figure 3. Streams and triage categories (registered secondary aim)

In [ ]:
copy_fig("main", "Figure 3", DLNM / "streams_triage.png", "Figure_3_streams_triage.png",
         "Primary models refitted for each ED stream and Manchester triage category. Cells: RR with √κ-widened 95% CI; "
         "bold with * = CI excludes 1; – = fewer than 2 arrivals per day (not modelled). Temperature: 99th or 1st percentile vs "
         "median, same day or cumulative over lags 0–21; ILI: 10th to 90th percentile; PM: per 10 µg/m³. Estimates, not tests (registered secondary aim).")

### Figure 4. Sensitivity analyses and negative controls

Redrawn from the saved estimates: each primary estimate (filled), its sensitivity analyses, and for ILI, PM10 and PM2.5 the negative-control outcomes (orange). Nominal 95% CIs (the primary estimate also nominal, for comparability).

In [ ]:
def panel_rows(h, est):
    s = sens.loc[[k for k in sens.index if k.split(" ", 1)[0] == h]]
    labels = [k.split(" ", 1)[1] for k in s.index]
    out = [(lab, s.loc[k, f"RR{est}"], s.loc[k, f"lo{est}"], s.loc[k, f"hi{est}"], "primary" if lab == "primary" else "sens")
           for lab, k in zip(labels, s.index)]
    if h != "TEMP":
        for nc, lab in (("KPP", "neg. control: injuries (unit)"), ("ICD S/T", "neg. control: injuries (diagnosis)")):
            n = neg.loc[f"{h} → injuries ({nc})"]
            out.append((lab, n["RR"], n["lo"], n["hi"], "nc"))
    return out


PANELS = [("TEMP", "_heat_lag0", "Temperature: heat, same day\n(99th percentile vs median)"),
          ("TEMP", "_cold_lag0", "Temperature: cold, same day\n(1st percentile vs median)"),
          ("ILI", "", "ILI → medical, cumulative\n(10th to 90th percentile)"),
          ("PM10", "", "PM10 → medical, cumulative\n(per 10 µg/m³)"),
          ("PM25", "", "PM2.5 → medical, cumulative\n(per 10 µg/m³)")]
fig, axes = plt.subplots(1, 5, figsize=(20, 7.5), gridspec_kw={"wspace": 1.15})
forest_rows = []
for ax, (h, est, title) in zip(axes, PANELS):
    rows_ = panel_rows(h, est)
    y = np.arange(len(rows_))[::-1]
    for yi, (lab, rr, lo, hi, kind) in zip(y, rows_):
        col = COLORS["orange"] if kind == "nc" else COLORS["blue"]
        ax.errorbar([rr], [yi], xerr=[[rr - lo], [hi - rr]], fmt="o", color=col, ms=5 if kind == "primary" else 3.5,
                    mfc=col if kind != "sens" else "white", lw=1.2)
        forest_rows.append({"panel": title.replace("\n", " "), "row": lab, "RR": rr, "lo": lo, "hi": hi, "type": kind})
    ax.axvline(1, color=COLORS["ink"], lw=0.6)
    ax.set_yticks(y, [textwrap.fill(r[0], 22) for r in rows_], fontsize=7)
    ax.set_title(title, loc="left", fontsize=9); ax.set_xlabel("RR (nominal 95% CI)", fontsize=8)
f4 = MAIN / "Figure_4_sensitivity_negative_controls.png"
fig.savefig(f4, bbox_inches="tight"); plt.show()
register("main", "Figure 4", f4, "Sensitivity analyses and negative controls. Filled blue: primary estimate; open blue: sensitivity analyses "
         "(each changes one element of the primary model); orange: negative-control outcomes (injuries). Nominal 95% CIs.",
         "outputs/03_2_dlnm/sensitivity.csv, negative_controls.csv")
pd.DataFrame(forest_rows).to_csv(MAIN / "Figure_4_data.csv", index=False)

## 3. Supplement

In [ ]:
# Table S1: variables of the primary analyses and negative controls
keep = list(codebook.index[codebook["use"] == "primary"]) + ["arr_injury_mts_all", "arr_dx_injury_icd", "triage_break_2024_06",
                                                            "arr_disease_mts_all", "arr_inp_mts_all", "arr_fasttrack_mts_all"]
s1 = codebook.loc[keep, ["definition", "unit", "source", "use", "analysis", "valid_from", "valid_to", "pct_missing", "note"]]
save_table("supplement", "Table S1", s1, "Table_S1_variables.csv",
           "Variables of the primary analyses, the negative-control outcomes and the stream outcomes (from the panel codebook).",
           "data/processed/daily_panel_codebook.csv", index=True)

# Table S2: model specification
LAG = lambda b, L: f"0–{L} days, " + ("one coefficient per day" if b[0] == "integer" else f"natural spline, {b[1]} log-spaced knots")
EXTRA = {"flu_adj": "influenza index (log 1 + x)", "hp_era": "referral era (break 2022-11-19)",
         "temp_cb": "temperature cross-basis (spline × lags 0–21)"}
s2 = pd.DataFrame([{
    "Analysis": ANALYSES[h], "Outcome": v["outcome"], "Exposure": v["exposure"] + (" (log 1 + x)" if v["transform"] == "log1p" else ""),
    "Exposure–response": "natural spline, knots at 10th/75th/90th pct" if v["exp_basis"] == "ns" else "linear",
    "Lags": LAG(v["lag_basis"], v["lag_max"]),
    "Adjustment": "time spline 7 df/year; day of week; holidays; COVID-19 period; " + "; ".join(EXTRA[e] for e in v["extra"]),
    "Lag-1 residual term": "yes" if str(settings["residual_decision"][h]) == "True" else "no",
} for h, v in settings["primary"].items()])
save_table("supplement", "Table S2", s2, "Table_S2_model_specification.csv",
           "Specification of the four primary DLNM models (quasi-Poisson; fixed before the first fit, deviations log items 31 and 43).",
           "outputs/03_2_dlnm/run_info.json")

copy_fig("supplement", "Figure S1", DLNM / "implementation_check.png", "Figure_S1_implementation_check.png",
         "Implementation check of the Python DLNM on simulated arrivals with known effects.")
copy_fig("supplement", "Figure S2", DLNM / "python_vs_r.png", "Figure_S2_python_vs_r.png",
         "Python DLNM vs R package dlnm on the real data: the same estimates and confidence intervals.")
rv = pd.read_csv(DLNM / "r_validation.csv")
save_table("supplement", "Table S3", rv, "Table_S3_r_validation.csv",
           f"Agreement between the Python DLNM and R dlnm ({runs['03_2'].get('r_version', 'R')}): maximum differences in log-RR and SE, F statistics and p-values.",
           "outputs/03_2_dlnm/r_validation.csv")

dummy = pd.read_csv(DLNM / "dummy_exposure.csv")
s4 = pd.DataFrame({h: {"dummy exposures": int((dummy["analysis"] == h).sum()),
                       "share with nominal p < 0.05": f"{(dummy.loc[dummy['analysis'] == h, 'p'] < 0.05).mean():.3f}",
                       "κ (mean dummy F)": f"{prim.loc[h, 'kappa']:.3f}", "CI widening √κ": f"{np.sqrt(prim.loc[h, 'kappa']):.3f}"}
                   for h in ANALYSES}).T.rename(index=ANALYSES)
save_table("supplement", "Table S4", s4, "Table_S4_calibration.csv",
           "Dummy-exposure calibration: share of 1,000 dummy exposures with nominal p < 0.05 (5% if the nominal test were calibrated), "
           "κ and the resulting widening of confidence intervals.", "outputs/03_2_dlnm/dummy_exposure.csv", index=True)
copy_fig("supplement", "Figure S3", DLNM / "dummy_exposure_qq.png", "Figure_S3_dummy_qq.png",
         "QQ plots of the nominal p-values of the dummy exposures for each primary analysis.")

cols = ["n_days", "RR_heat_lag0", "lo_heat_lag0", "hi_heat_lag0", "RR_cold_lag0", "lo_cold_lag0", "hi_cold_lag0",
        "RR_heat_vs_median", "lo_heat_vs_median", "hi_heat_vs_median", "RR_cold_vs_median", "lo_cold_vs_median", "hi_cold_vs_median",
        "RR", "lo", "hi", "increment", "p"]
s5 = sens[[c for c in cols if c in sens]].copy()
s5.insert(0, "analysis", [ANALYSES[k.split(" ", 1)[0]] for k in s5.index])
s5.index = [k.split(" ", 1)[1] for k in s5.index]
s5 = s5.rename_axis("variant").reset_index()
num = s5.select_dtypes("number").columns.drop(["n_days", "p"], errors="ignore")
s5[num] = s5[num].round(3); s5["p"] = s5["p"].map(fmt_p)
save_table("supplement", "Table S5", s5, "Table_S5_sensitivity.csv",
           "All sensitivity analyses: RR with nominal 95% CI and nominal p. Registered: spline df, lag length, excluding the pandemic "
           "period; the others were added (deviations log items 36–37).", "outputs/03_2_dlnm/sensitivity.csv")
qa = pd.read_csv(DLNM / "qaic.csv", index_col=0).rename(index=ANALYSES)
save_table("supplement", "Table S6", qa.round(1), "Table_S6_qaic.csv",
           "Quasi-AIC of the time spline (4, 7, 10 df per year) on the same days with a common dispersion; 7 df/year was prespecified and kept.",
           "outputs/03_2_dlnm/qaic.csv", index=True)
copy_fig("supplement", "Figure S4", DLNM / "model_checks.png", "Figure_S4_model_checks.png",
         "Residual checks of the primary models (residuals over time, autocorrelation).")
copy_fig("supplement", "Figure S5", DLNM / "ILI_PM10_PM25_lag_response.png", "Figure_S5_lag_response.png",
         "Lag–response of ILI, PM10 and PM2.5 (√κ-widened 95% CI).")
copy_fig("supplement", "Figure S6", DLNM / "TEMP_dose_response_by_stream.png", "Figure_S6_temperature_by_stream.png",
         "Temperature–response by stream: cumulative over lags 0–21 (blue, 95% CI) and same day (red).")

In [ ]:
# Figure S7: exposure-response curves of the primary exposures (spline vs linear), redrawn from saved estimates
dr = pd.read_csv(DLNM / "dose_response.csv")
drt = pd.read_csv(DLNM / "dose_response_tests.csv", index_col=0)
CURVES = ["Temperature → all arrivals", "ILI → medical", "PM10 → medical", "PM2.5 → medical",
          "PM10 → injuries (control)", "PM2.5 → injuries (control)"]
XLAB = {"Temperature → all arrivals": "daily mean temperature (°C)", "ILI → medical": "ILI rate per 100,000 (log scale)",
        "PM10 → medical": "PM10 (µg/m³)", "PM2.5 → medical": "PM2.5 (µg/m³)",
        "PM10 → injuries (control)": "PM10 (µg/m³)", "PM2.5 → injuries (control)": "PM2.5 (µg/m³)"}
fig, axes = plt.subplots(2, 3, figsize=(14, 8), gridspec_kw={"hspace": 0.5, "wspace": 0.25})
for ax, c in zip(axes.flat, CURVES):
    sp, li = dr[(dr["curve"] == c) & (dr["model"] == "spline")], dr[(dr["curve"] == c) & (dr["model"] == "linear")]
    ax.fill_between(sp["exposure_value"], sp["lo"], sp["hi"], color=COLORS["blue"], alpha=0.2, lw=0)
    ax.plot(sp["exposure_value"], sp["RR"], color=COLORS["blue"], lw=2, label="spline")
    ax.plot(li["exposure_value"], li["RR"], color=COLORS["orange"], lw=1.3, ls="--", label="linear")
    ax.axhline(1, color=COLORS["ink"], lw=0.6)
    if c.startswith("ILI"):
        ax.set_xscale("symlog", linthresh=1.0); ax.set_xlim(0, sp["exposure_value"].max() * 1.05)
    t = drt.loc[c]
    ax.set_title(f"{c}\nnonlinearity p = {fmt_p(t['p nonlinearity (nominal)'])}", loc="left", fontsize=8.5)
    ax.set_ylabel("cumulative RR vs reference", fontsize=8)
    ax.set_xlabel(XLAB[c], fontsize=8)
axes.flat[0].legend(fontsize=7)
fs7 = SUPP / "Figure_S7_exposure_response.png"
fig.savefig(fs7, bbox_inches="tight"); plt.show()
register("supplement", "Figure S7", fs7, "Exposure–response curves of the primary exposures: natural spline (blue, √κ-widened 95% CI) vs "
         "the linear model (orange); cumulative over all lags; nominal F-test of nonlinearity. References: median temperature, 10th "
         "percentile of ILI and PM. Exploratory (deviations log item 42).", "outputs/03_2_dlnm/dose_response.csv")
save_table("supplement", "Table S7", drt.loc[CURVES].round(4), "Table_S7_exposure_response_tests.csv",
           "Tests of the exposure–response curves in Figure S7 (nominal).", "outputs/03_2_dlnm/dose_response_tests.csv", index=True)

# Table S8: streams x triage categories, all estimates
st = pd.read_csv(DLNM / "streams_triage_long.csv")
st["RR (95% CI)"] = [rr_ci(a, b, c) for a, b, c in zip(st["RR"], st["lo"], st["hi"])]
st["estimate"] = st["estimate"].str.replace("TEMP", "Temperature:", regex=False)
s8 = st.pivot_table(index=["estimate", "stream"], columns="triage", values="RR (95% CI)", aggfunc="first", sort=False)
s8 = s8.reindex(columns=[c for c in ["all", "1 red", "2 orange", "3 yellow", "4 green", "5 blue"] if c in s8.columns]).fillna("–")
save_table("supplement", "Table S8", s8, "Table_S8_streams_triage.csv",
           "All estimates of Figure 3: RR (√κ-widened 95% CI) by stream and triage category; – = fewer than 2 arrivals per day.",
           "outputs/03_2_dlnm/streams_triage_long.csv", index=True)

pm = pd.read_csv(DLNM / "pm25_extra.csv", index_col=0)
save_table("supplement", "Table S9", pm[["RR", "lo", "hi", "increment", "p", "n_days"]].round(4), "Table_S9_pm25_extra.csv",
           "PM2.5 to all arrivals, and the coarse fraction (PM10 − PM2.5) to medical arrivals (nominal 95% CI).",
           "outputs/03_2_dlnm/pm25_extra.csv", index=True)

# Cross-correlation (complementary analysis) and comparison
copy_fig("supplement", "Figure S8", CCF / "ccf_main_and_negative_control.png", "Figure_S8_ccf_main.png",
         "Prewhitened cross-correlations of the primary analyses with negative-control outcomes; pointwise and simultaneous 95% limits.")
copy_fig("supplement", "Figure S9", CCF / "ccf_sensitivity.png", "Figure_S9_ccf_sensitivity.png",
         "Cross-correlation sensitivity analyses.")
copy_fig("supplement", "Figure S10", CMP / "timing_ccf_vs_dlnm.png", "Figure_S10_ccf_vs_dlnm_timing.png",
         "Timing of the associations: cross-correlation by lag vs DLNM lag-specific RR.")
copy_fig("supplement", "Figure S11", CCF / "exposure_spearman.png", "Figure_S11_exposure_correlations.png",
         "Spearman correlations between exposures, raw and after removing seasonality.")
save_table("supplement", "Table S10", pd.read_csv(CMP / "verdict.csv"), "Table_S10_ccf_vs_dlnm.csv",
           "Cross-correlation vs DLNM for the four primary analyses.", "outputs/03_3_comparison/verdict.csv")
save_table("supplement", "Table S11", pd.read_csv(CMP / "matched_sensitivity.csv"), "Table_S11_matched_sensitivity.csv",
           "Sensitivity analyses run with both methods: agreement on signal / no signal.", "outputs/03_3_comparison/matched_sensitivity.csv")

if DEVIATIONS.exists():
    dst = SUPP / "Appendix_deviations_log.md"
    shutil.copy2(DEVIATIONS, dst)
    register("supplement", "Appendix", dst, "Deviations from the preregistration (all items).", "docs/deviations_log.md")
else:
    print("docs/deviations_log.md not found: appendix skipped")

## 4. Index and workbook

In [ ]:
index = pd.DataFrame(INDEX)
with pd.ExcelWriter(OUT / "article_tables.xlsx") as xw:
    for item, df in TABLES.items():
        df.to_excel(xw, sheet_name=item.replace(" ", "_")[:31], index=False)

lines = ["# Figures and tables for the main article", "",
         f"Built by `notebooks/04_article_outputs.ipynb` from the outputs of notebooks 01–03 (panel `{sha[:12]}`). "
         "Rebuilt on each run: do not edit files here. Captions are drafts.", "",
         "All tables are also in `article_tables.xlsx` (one sheet each). Figures are PNG at 300 dpi.", "",
         "Not included (other papers or exploratory): individual holidays, hospital occupancy, Martinovo, moon, pollen, the "
         "exploratory cross-correlation screen. Still to prepare by hand: RECORD and time-series reporting checklists.", ""]
for part, title in (("main", "Main text"), ("supplement", "Supplement")):
    lines += [f"## {title}", "", "| Item | File | Draft caption | Source |", "|---|---|---|---|"]
    lines += [f"| {r['item']} | `{r['file']}` | {r['caption']} | `{r['source']}` |" for _, r in index[index["part"] == part].iterrows()]
    lines.append("")
(OUT / "README.md").write_text("\n".join(lines), encoding="utf-8")
print(f"{len(index)} items -> {OUT.relative_to(ROOT)}")
display(index[["part", "item", "file"]])